In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
import requests
import io

# 1. Scrape the current S&P 500 list from Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
	"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(response.text))
df_sp500 = tables[0]

# 2. Prepare Tickers
def clean_ticker(ticker):
	return ticker.replace('.', '-')

symbols = df_sp500['Symbol'].apply(clean_ticker).tolist()

# 3. Download Latest Price Data
print(f"Downloading data for {len(symbols)} symbols...")
price_data = yf.download(
	symbols,
	period="1d",
	group_by='ticker',
	auto_adjust=False,
	progress=False
)

# 4. Extract Closing Prices and filter out delisted/missing ones
# This creates the 'valid_prices' variable for the next cell
close_prices = price_data.xs("Close", level=1, axis=1).iloc[-1]
valid_prices = close_prices.dropna()

print(f"Successfully retrieved prices for {len(valid_prices)} stocks.")

Successfully retrieved prices for 503 stocks.


In [3]:
market_caps = {}
total = len(valid_prices)

print(f"Fetching Market Cap data for {total} tickers (this will take a few minutes)...")
for index, ticker in enumerate(valid_prices.index):
	if index % 25 == 0:
		print(f"Progress: {index}/{total}")
	try:
		t = yf.Ticker(ticker)
		# .info is slower but contains the marketCap field
		market_caps[ticker] = t.info.get("marketCap", np.nan)
	except:
		market_caps[ticker] = np.nan

print("Market Cap collection complete.")

Fetching Market Cap data for 503 tickers (this will take a few minutes)...
Progress: 0/503
Progress: 25/503
Progress: 50/503
Progress: 75/503
Progress: 100/503
Progress: 125/503
Progress: 150/503
Progress: 175/503
Progress: 200/503
Progress: 225/503
Progress: 250/503
Progress: 275/503
Progress: 300/503
Progress: 325/503
Progress: 350/503
Progress: 375/503
Progress: 400/503
Progress: 425/503
Progress: 450/503
Progress: 475/503
Progress: 500/503
Market Cap collection complete.


In [4]:
# 1. Input Portfolio Size
while True:
	try:
		portfolio_size = float(input("Enter the total value of your portfolio: "))
		break
	except ValueError:
		print("Invalid input. Please enter a number.")

# 2. Build the Trade Table
final_df = pd.DataFrame({
	"Ticker": close_prices.index,
	"Price": close_prices.values,
	"Market Capitalization": close_prices.index.map(market_caps)
})

# 3. Calculate Equal Weighting
position_size = portfolio_size / len(final_df)

# 4. Calculate Number of Shares (REMOVED math.floor for fractional shares)
# We round to 4 decimal places for readability in Excel
final_df["Number of Shares to Buy"] = (position_size / final_df["Price"]).round(4)

# 5. Final Clean and Export
final_df.to_excel("sp500_fractional_shares_strategy.xlsx", index=False)

print(f"\nSuccess! Strategy with fractional shares saved for {len(final_df)} stocks.")
final_df.head()


Success! Strategy with fractional shares saved for 503 stocks.


,Ticker,Price,Market Capitalization,Number of Shares to Buy
0,KHC,23.555000,27892844544,0.0084
1,MRNA,47.520000,18585243648,0.0042
2,LRCX,222.990005,281172377600,0.0009
3,USB,55.924999,86955597824,0.0036
4,CEG,285.380005,103375847424,0.0007
